In [1]:
def distribution_trans_experiment(dataframe, target_column, trunc_top=None, trunc_bottom=None):
    '''
    This function creates a copy of a dataframe and checks to see which transformations
    work best with it. In addition, it optionally truncates the dataframe.
    NOTE: 
    You have to give both trunc_top & trunc_bottom. I'll update this later

    param:               dataframe (pd.DataFrame)
    param:               target_column (pd.Dataframe column)
    param(optional):     trunc_top (0-1)
    param(optional):     trunc_bottom (0-1)
    '''

    import numpy as np
    from scipy import stats
    from sklearn.preprocessing import PowerTransformer, QuantileTransformer
    import matplotlib.pyplot as plt
    import seaborn as sns
    import pandas as pd
    import matplotlib.pyplot as plt
    
    # Create a copy of the dataframe
    df_copy = dataframe.copy()

    # Set the trunctation amount up top if you want it.
    if trunc_top is not None and trunc_bottom is not None:
        df_copy = df_copy[
            (df_copy[target_column] >= df_copy[target_column].quantile(q=trunc_bottom)) &
            (df_copy[target_column] <= df_copy[target_column].quantile(q=trunc_top))
        ]

    # Basic transformations
    df_copy['target_sqrt'] = np.sqrt(df_copy[target_column])
    df_copy['target_cbrt'] = np.cbrt(df_copy[target_column])

    # Log 
    df_copy['target_log'] = np.log1p(df_copy[target_column])  # log(1 + x) to handle zeros

    # Boxcox
    df_copy['target_boxcox'], _ = stats.boxcox(df_copy[target_column] + 1)  # +1 if zeros exist

    # Yeo 
    pt = PowerTransformer(method='yeo-johnson')
    df_copy['target_yeojohnson'] = pt.fit_transform(df_copy[[target_column]])

    # Quantile
    qt = QuantileTransformer(output_distribution='normal')
    df_copy['target_quantile'] = qt.fit_transform(df_copy[[target_column]])

    # Bin
    df_copy['target_bin'] = pd.qcut(df_copy[target_column], q=4, labels=False)

    # Create a subplot visual.
    fig, axes = plt.subplots(1, 6, figsize=(30, 2))
    sns.histplot(df_copy, x='target_sqrt', ax=axes[0])
    sns.histplot(df_copy, x='target_log', ax=axes[1])
    sns.histplot(df_copy, x='target_boxcox', ax=axes[2])
    sns.histplot(df_copy, x='target_yeojohnson', ax=axes[3])
    sns.histplot(df_copy, x='target_quantile', ax=axes[4])
    sns.histplot(df_copy, x='target_bin', ax=axes[5])

    # Shows the plots
    plt.show()

In [2]:
# Let's check to see what works on this target variable.
distribution_trans_experiment(df,"Kilograms",.95, .05)

NameError: name 'df' is not defined